# DocuMind AI — 02. OCR Evaluation & Geometry Normalization
This notebook evaluates character and word error metrics and demonstrates relative bounding box normalization for frontend UI alignment.


In [ ]:
import sys
sys.path.append('../..')

from backend.app.services.ocr_service import ocr_service
from backend.app.utils.pdf_tools import extract_pdf_native_spans
from pathlib import Path


## 1. Character Error Rate (CER) and Word Error Rate (WER) Logic


In [ ]:
def levenshtein_distance(s1, s2):
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]

def calculate_cer(reference, hypothesis):
    d = levenshtein_distance(reference, hypothesis)
    return d / max(1, len(reference))

# Demonstration
ref = "TOTAL AMOUNT: INR 11,800.00"
hyp = "TOTAL AMOUNT: INR 11,800.00"
print(f"CER: {calculate_cer(ref, hyp):.4f} (Exact match)")

hyp_noise = "TOTAL AM0UNT: INR 11,800.00"
print(f"CER with OCR noise: {calculate_cer(ref, hyp_noise):.4f}")


## 2. Coordinate Normalization to [0, 1] Range


In [ ]:
pdf_path = Path('../../data/sample_invoices/01_clean_standard_invoice.pdf')
if pdf_path.exists():
    tokens = extract_pdf_native_spans(pdf_path)
    print(f"Extracted {len(tokens)} normalized vector tokens.")
    for t in tokens[:5]:
        b = t['bbox']
        print(f"Token: '{t['text']}' | Normalized BBox: x1={b['x1']}, y1={b['y1']}, x2={b['x2']}, y2={b['y2']}")
else:
    print("Sample PDF not found. Generate samples via backend.app.utils.generate_samples first.")
